# SupeRR Selector Season 2 — Comprehensive Dataset EDA & Overview
> **Purpose**: A single, self-contained reference notebook providing a complete structural overview of all **6 core CSV files** in the dataset:
> - **File inventory & size stats**
> - **Column names, data types, null counts, unique values, and samples**
> - **Key categorical & distribution overviews** for each dataset
> - **Cross-table relational joins & key overlap validation**
>
> *(Note: files in `pressure_feasibility_output` are excluded to focus strictly on raw organizer data)*

## 1. Setup & Environment
Configure paths and pandas display parameters for readability.

In [1]:
import os, sys, glob
from pathlib import Path
import pandas as pd
import numpy as np

# Adjust display settings for clean tabular viewing
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

# Dataset folder resolution (relative with fallback to absolute)
DATA_DIR = Path("SupeRR Selector Season 2 | Data Pack")
if not DATA_DIR.exists():
    DATA_DIR = Path("/Users/vansh/Downloads/SupeRR Selector Season 2 | Problem Statement/SupeRR Selector Season 2 | Data Pack")

print(f"Dataset directory: {DATA_DIR.resolve()}")
print(f"Directory exists: {DATA_DIR.is_dir()}")

Dataset directory: /Users/vansh/Downloads/SupeRR Selector Season 2 | Problem Statement/SupeRR Selector Season 2 | Data Pack
Directory exists: True


## 2. High-Level File Inventory
Summary of all 6 core CSV files: row counts, column counts, file size on disk, and memory usage.

In [2]:
core_files = [
    "competitions.csv",
    "teams.csv",
    "matches.csv",
    "playing_eleven.csv",
    "competition_team_player_mapping.csv",
    "ball_by_ball.csv"
]

inventory = []
for fname in core_files:
    fpath = DATA_DIR / fname
    if fpath.exists():
        size_mb = round(fpath.stat().st_size / (1024 * 1024), 2)
        with open(fpath, 'r', encoding='utf-8-sig', errors='replace') as f:
            total_lines = sum(1 for _ in f) - 1 # minus header
        df_head = pd.read_csv(fpath, nrows=2)
        inventory.append({
            "File Name": fname,
            "Size (MB)": size_mb,
            "Rows": total_lines,
            "Cols": len(df_head.columns),
            "Sample Columns": ", ".join(list(df_head.columns)[:5]) + ("..." if len(df_head.columns) > 5 else "")
        })

df_inventory = pd.DataFrame(inventory)
display(df_inventory)

,File Name,Size (MB),Rows,Cols,Sample Columns
0,competitions.csv,0.01,141,4,"comp_id, comp_name, match_format, season"
1,teams.csv,0.01,353,3,"comp_id, team_id, team_name"
2,matches.csv,0.12,883,22,"match_id, comp_id, match_number, match_date, s..."
3,playing_eleven.csv,0.92,28381,9,"match_id, team_id, player_id, batting_order, i..."
4,competition_team_player_mapping.csv,0.44,6413,6,"comp_id, comp_name, team_id, team_name, player..."
5,ball_by_ball.csv,39.17,200999,33,"match_id, innings, overid, over_ball_num, inni..."


## 3. General EDA Helper: Schema & Column Inspector
This helper function outputs a detailed breakdown for any file:
- Total rows, columns, memory usage
- Per-column: Data type, Non-Null count, Null count, Missing %, Unique values count, and Sample/Top values
- First 3 rows of data

In [3]:
def inspect_df(df: pd.DataFrame, title: str = "Dataset Overview") -> pd.DataFrame:
    """Displays concise schema summary and returns column-level statistics table."""
    total_rows = len(df)
    mem_mb = round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2)
    print("=" * 90)
    print(f"  {title.upper()} | {total_rows:,} rows x {df.shape[1]} columns | Memory: {mem_mb} MB")
    print("=" * 90)
    
    col_stats = []
    for col in df.columns:
        s = df[col]
        null_count = s.isna().sum()
        null_pct = round((null_count / total_rows) * 100, 2) if total_rows else 0.0
        n_unique = s.nunique(dropna=True)
        
        # Sample or top value representation
        if n_unique > 0:
            top_vals = s.dropna().value_counts().head(2).to_dict()
            sample_repr = "; ".join([f"{str(k)[:20]} ({v})" for k, v in top_vals.items()])
        else:
            sample_repr = "<ALL NULL>"
            
        col_stats.append({
            "Column": col,
            "Type": str(s.dtype),
            "Non-Null": total_rows - null_count,
            "Null": null_count,
            "Null %": null_pct,
            "Unique": n_unique,
            "Top Values (Freq)": sample_repr
        })
        
    summary_df = pd.DataFrame(col_stats)
    display(summary_df)
    print("\n--- First 3 Rows ---")
    display(df.head(3))
    return summary_df

---
## 4. Per-File Overviews

### 4.1 `competitions.csv`
Defines tournaments, match formats (e.g. T20, T20I), season years, and competition names.

In [4]:
df_comps = pd.read_csv(DATA_DIR / "competitions.csv")
inspect_df(df_comps, "competitions.csv")

print("\n--- Match Formats Breakdown ---")
display(df_comps['match_format'].value_counts(dropna=False).to_frame(name='Competitions'))

print("\n--- Seasons Covered ---")
display(df_comps['season'].value_counts(dropna=False).head(10).to_frame(name='Competitions'))

  COMPETITIONS.CSV | 141 rows x 4 columns | Memory: 0.03 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,comp_id,int64,141,0,0.0,141,3395 (1); 2755 (1)
1,comp_name,object,141,0,0.0,141,South Africa in New (1); Sri Lanka in New Zea...
2,match_format,object,141,0,0.0,2,T20I (134); T20 (7)
3,season,object,141,0,0.0,8,2024 (33); 2025 (33)



--- First 3 Rows ---


,comp_id,comp_name,match_format,season
0,3395,South Africa in New Zealand T20I Series 2026,T20I,2026
1,3405,Afghanistan v Sri Lanka T20I Series 2026,T20I,2026
2,3465,India in Zimbabwe T20I Series 2026,T20I,2026



--- Match Formats Breakdown ---


,Competitions
match_format,
T20I,134
T20,7



--- Seasons Covered ---


,Competitions
season,
2024,33
2025,33
2026,25
2022,25
2023,20
2022/23,3
2023/24,1
2024/25,1


### 4.2 `teams.csv`
Defines all teams associated with competitions (`comp_id` to `team_id`).

In [5]:
df_teams = pd.read_csv(DATA_DIR / "teams.csv")
inspect_df(df_teams, "teams.csv")

print(f"Total Unique Teams: {df_teams['team_id'].nunique()}")
print(f"Total Unique Competitions in teams: {df_teams['comp_id'].nunique()}")
print("\n--- Teams per Competition (Top 10) ---")
display(df_teams.groupby('comp_id')['team_id'].count().sort_values(ascending=False).head(10).to_frame(name='Team Count'))

  TEAMS.CSV | 353 rows x 3 columns | Memory: 0.03 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,comp_id,int64,353,0,0.0,141,1358 (10); 1715 (10)
1,team_id,int64,353,0,0.0,38,18 (33); 21 (26)
2,team_name,object,353,0,0.0,38,India (33); Bangladesh (26)



--- First 3 Rows ---


,comp_id,team_id,team_name
0,1331,22,England
1,1331,28,West Indies
2,1335,18,India


Total Unique Teams: 38
Total Unique Competitions in teams: 141

--- Teams per Competition (Top 10) ---


,Team Count
comp_id,
1358,10
1715,10
3298,10
2193,10
2725,10
3133,8
2670,8
3289,8
1533,6


### 4.3 `matches.csv`
Metadata for each match: tournament ID, teams, dates, venue, stage, toss, result, and DLS/revised overs information.

In [6]:
df_matches = pd.read_csv(DATA_DIR / "matches.csv")
inspect_df(df_matches, "matches.csv")

print("\n--- Match Stages Distribution ---")
display(df_matches['stage'].value_counts(dropna=False).to_frame(name='Matches'))

print("\n--- Match Knockout Flag ---")
display(df_matches['knock_out'].value_counts(dropna=False).to_frame(name='Matches'))

print("\n--- Result Types (Top 10) ---")
display(df_matches['result'].value_counts(dropna=False).head(10).to_frame(name='Matches'))

revised_cnt = df_matches['revised_target'].notna().sum()
print(f"\nMatches with revised target (DLS / Rain): {revised_cnt} out of {len(df_matches)} ({round(revised_cnt/len(df_matches)*100, 2)}%)")

  MATCHES.CSV | 883 rows x 22 columns | Memory: 0.46 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,match_id,int64,883,0,0.00,883,13511 (1); 40761 (1)
1,comp_id,int64,883,0,0.00,138,2725 (75); 1715 (74)
2,match_number,int64,883,0,0.00,75,2 (137); 1 (136)
3,match_date,object,883,0,0.00,686,2024-05-12 (4); 2025-03-23 (3)
4,stage,object,883,0,0.00,7,League (620); Others (226)
5,venue_id,int64,883,0,0.00,98,3 (51); 49 (39)
6,venue_country,object,883,0,0.00,17,India (425); United Arab Emirates (69)
7,home_team_id,int64,883,0,0.00,36,18 (62); 28 (51)
8,home_team,object,883,0,0.00,36,India (62); West Indies (51)
9,away_team_id,int64,883,0,0.00,36,18 (58); 25 (53)



--- First 3 Rows ---


,match_id,comp_id,match_number,match_date,stage,venue_id,venue_country,home_team_id,home_team,away_team_id,away_team,home_team_captain_id,away_team_captain_id,toss_win_team_id,toss_opted,man_of_the_match,first_inn_revised_overs,second_inn_revised_overs,revised_target,knock_out,win_team_id,result
0,13511,1331,1,2022-01-22,League,54,West Indies,28,West Indies,22,England,NaN,NaN,28.0,Fielding,0,NaN,NaN,NaN,0,28.0,West Indies Won by 9 Wickets
1,13512,1331,2,2022-01-23,League,54,West Indies,28,West Indies,22,England,494.0,478.0,28.0,Fielding,0,NaN,NaN,NaN,0,22.0,England Won by 1 Run
2,13513,1331,3,2022-01-26,League,54,West Indies,28,West Indies,22,England,494.0,1596.0,22.0,Fielding,4986,NaN,NaN,NaN,0,28.0,West Indies Won by 20 runs



--- Match Stages Distribution ---


,Matches
stage,
League,620
Others,226
Final,12
Qualifier,10
Super 4,6
Eliminator,5
Semi Final,4



--- Match Knockout Flag ---


,Matches
knock_out,
0,852
1,31



--- Result Types (Top 10) ---


,Matches
result,
No Result,21
Match Abandoned,19
India Won by 7 wickets,8
Delhi Capitals Won by 6 wickets,7
Kolkata Knight Riders Won by 8 wickets,6
India Won by 6 Wickets,5
Gujarat Titans Won by 7 wickets,5
Punjab Kings Won by 5 wickets,5
Royal Challengers Bengaluru Won by 6 wickets,4



Matches with revised target (DLS / Rain): 23 out of 883 (2.6%)


### 4.4 `playing_eleven.csv`
Specifies the playing XI roster (match_id, team_id, player_id) for each game.

In [7]:
df_xi = pd.read_csv(DATA_DIR / "playing_eleven.csv")
inspect_df(df_xi, "playing_eleven.csv")

print(f"Unique Matches in Playing XI: {df_xi['match_id'].nunique()}")
print(f"Unique Players in Playing XI: {df_xi['player_id'].nunique()}")

# Validate players per team per match
players_per_team_match = df_xi.groupby(['match_id', 'team_id'])['player_id'].count()
print("\n--- Players per Team per Match Distribution ---")
display(players_per_team_match.value_counts().to_frame(name='Frequency'))

  PLAYING_ELEVEN.CSV | 28,381 rows x 9 columns | Memory: 3.21 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,match_id,int64,28381,0,0.00,880,40754 (51); 31243 (51)
1,team_id,int64,28381,0,0.00,38,18 (1661); 3508 (1532)
2,player_id,int64,28381,0,0.00,1082,6464 (167); 2876 (155)
3,batting_order,float64,19703,8678,30.58,13,3.0 (1744); 1.0 (1743)
4,is_played,bool,28381,0,0.00,2,True (19714); False (8667)
5,is_captain,object,19714,8667,30.54,2,False (18031); True (1683)
6,is_wk,object,19714,8667,30.54,2,False (18039); True (1675)
7,is_impact,int64,28381,0,0.00,2,0 (27823); 1 (558)
8,is_impact_off,int64,28381,0,0.00,2,0 (27823); 1 (558)



--- First 3 Rows ---


,match_id,team_id,player_id,batting_order,is_played,is_captain,is_wk,is_impact,is_impact_off
0,13511,22,1187,1.0,True,False,False,0,0
1,13511,22,6273,2.0,True,False,False,0,0
2,13511,22,1210,3.0,True,False,False,0,0


Unique Matches in Playing XI: 880
Unique Players in Playing XI: 1082

--- Players per Team per Match Distribution ---


,Frequency
player_id,
11,569
25,250
15,215
16,162
12,141
17,91
20,63
24,56
23,48


### 4.5 `competition_team_player_mapping.csv`
Official squad mapping associating players with their respective teams and competitions.

In [8]:
df_mapping = pd.read_csv(DATA_DIR / "competition_team_player_mapping.csv")
inspect_df(df_mapping, "competition_team_player_mapping.csv")

print(f"Unique Players in Mapping: {df_mapping['player_id'].nunique()}")
print(f"Unique Competitions in Mapping: {df_mapping['comp_id'].nunique()}")
print(f"Unique Teams in Mapping: {df_mapping['team_id'].nunique()}")

  COMPETITION_TEAM_PLAYER_MAPPING.CSV | 6,413 rows x 6 columns | Memory: 1.39 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,comp_id,int64,6413,0,0.0,140,3298 (275); 1715 (264)
1,comp_name,object,6413,0,0.0,140,IPL 2026 (275); IPL 2023 (264)
2,team_id,int64,6413,0,0.0,38,18 (551); 28 (415)
3,team_name,object,6413,0,0.0,38,India (551); West Indies (415)
4,player_id,int64,6413,0,0.0,1217,6464 (33); 2876 (30)
5,player_name,object,6413,0,0.0,1210,Arshdeep Singh (33); Axar Patel (30)



--- First 3 Rows ---


,comp_id,comp_name,team_id,team_name,player_id,player_name
0,1331,England in West Indies T20I Series 2022,22,England,478,Eoin Morgan
1,1331,England in West Indies T20I Series 2022,22,England,479,Adil Rashid
2,1331,England in West Indies T20I Series 2022,22,England,722,Liam Dawson


Unique Players in Mapping: 1217
Unique Competitions in Mapping: 140
Unique Teams in Mapping: 38


### 4.6 `ball_by_ball.csv`
The granular delivery-level event log (largest file: ~41 MB). Contains bowler, striker, runs, extras, boundaries, dismissals, and bowling styles.

In [9]:
df_bbb = pd.read_csv(DATA_DIR / "ball_by_ball.csv", low_memory=False)
inspect_df(df_bbb, "ball_by_ball.csv")

print("\n--- Innings Distribution ---")
display(df_bbb['innings'].value_counts(dropna=False).sort_index().to_frame(name='Deliveries'))

print("\n--- Batter Runs Breakdown ---")
display(df_bbb['runs'].value_counts(dropna=False).head(10).to_frame(name='Deliveries'))

print("\n--- Extras Type Breakdown ---")
display(df_bbb['extras_type'].value_counts(dropna=False).to_frame(name='Count'))

print("\n--- Dismissal Type Breakdown ---")
display(df_bbb['dismissal_type'].value_counts(dropna=False).head(10).to_frame(name='Count'))

if 'bowling_type' in df_bbb.columns:
    print("\n--- Bowling Type (Top 10) ---")
    display(df_bbb['bowling_type'].value_counts(dropna=False).head(10).to_frame(name='Count'))

  BALL_BY_BALL.CSV | 200,999 rows x 33 columns | Memory: 165.28 MB


,Column,Type,Non-Null,Null,Null %,Unique,Top Values (Freq)
0,match_id,int64,200999,0,0.00,867,36199 (267); 31248 (265)
1,innings,int64,200999,0,0.00,4,1 (105727); 2 (95234)
2,overid,int64,200999,0,0.00,20,1 (10818); 2 (10788)
3,over_ball_num,int64,200999,0,0.00,13,1 (32350); 2 (32257)
4,innings_ball_num,int64,200999,0,0.00,136,1 (1724); 2 (1721)
5,batting_team_id,int64,200999,0,0.00,38,18 (13369); 25 (10482)
6,bowling_team_id,int64,200999,0,0.00,38,18 (13394); 28 (10629)
7,striker_id,int64,200999,0,0.00,888,813 (2920); 1699 (2850)
8,striker_name,object,200999,0,0.00,885,Jos Buttler (2920); Suryakumar Yadav (2850)
9,non_striker_id,int64,200999,0,0.00,867,6288 (3269); 813 (2912)



--- First 3 Rows ---


,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,non_striker_name,bowler_id,bowler_name,batting_type,bowling_type,batting_order,runs,is_four,is_six,is_wicket,dismissal_type_id,dismissal_type,dismissed_player_id,extras_type,extras1_runs,extras_2_runs,pace_or_spin,bowl_speed,stroke,delivery_style,over_or_around,on_or_off,delivery_ist_time
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Cover Drive,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Flick,Inswing,Over,On-side,2022-01-23 00:00:00 UTC
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,Tom Banton,1772,Sheldon Cottrell,RHB,Left-arm fast-medium,1,0,False,False,False,NaN,NaN,NaN,NaN,6,0,Pace,0.0,Square Cut,Inswing,Over,Off-side,2022-01-23 00:00:00 UTC



--- Innings Distribution ---


,Deliveries
innings,
1,105727
2,95234
3,25
4,13


... (4 outputs left out)


--- Dismissal Type Breakdown ---


,Count
dismissal_type,
NaN,190118
Caught,7557
Bowled,1775
LBW,709
RunOut,600
Stump,214
Hit Wicket,17
Retire Out,6
Obstracting the field,2



--- Bowling Type (Top 10) ---


,Count
bowling_type,
Right-arm Leg-Break,28027
Right-arm medium-fast,27014
Right-arm Fast,26637
Right-arm medium,23919
Slow left-arm orthodox,23582
Right-arm offbreak,20948
Right-arm fast-medium,15380
Left-arm medium-fast,12492
Left-arm fast-medium,7780


---
## 5. Cross-Table Relational Join & ID Coverage Map
Verify referential integrity across the 6 files:
- Does every ball-by-ball record belong to a match in `matches.csv`?
- Are players in `ball_by_ball` present in `playing_eleven` and squad mappings?
- Are match team IDs valid in `teams.csv`?

In [10]:
match_ids_m = set(df_matches['match_id'].dropna().astype(str))
match_ids_bbb = set(df_bbb['match_id'].dropna().astype(str))
match_ids_xi = set(df_xi['match_id'].dropna().astype(str))

players_xi = set(df_xi['player_id'].dropna().astype(str))
players_map = set(df_mapping['player_id'].dropna().astype(str))
strikers_bbb = set(df_bbb['striker_id'].dropna().astype(str))
bowlers_bbb = set(df_bbb['bowler_id'].dropna().astype(str))
all_bbb_players = strikers_bbb | bowlers_bbb

team_ids_teams = set(df_teams['team_id'].dropna().astype(str))
team_ids_matches = set(df_matches['home_team_id'].dropna().astype(str)) | set(df_matches['away_team_id'].dropna().astype(str))

join_checks = [
    {
        "Join Relationship": "matches.match_id <-> ball_by_ball.match_id",
        "Left Count": len(match_ids_m),
        "Right Count": len(match_ids_bbb),
        "Intersection": len(match_ids_m & match_ids_bbb),
        "Missing on Right": len(match_ids_m - match_ids_bbb),
        "Orphan Right IDs": len(match_ids_bbb - match_ids_m)
    },
    {
        "Join Relationship": "matches.match_id <-> playing_eleven.match_id",
        "Left Count": len(match_ids_m),
        "Right Count": len(match_ids_xi),
        "Intersection": len(match_ids_m & match_ids_xi),
        "Missing on Right": len(match_ids_m - match_ids_xi),
        "Orphan Right IDs": len(match_ids_xi - match_ids_m)
    },
    {
        "Join Relationship": "playing_eleven.player_id <-> ball_by_ball players",
        "Left Count": len(players_xi),
        "Right Count": len(all_bbb_players),
        "Intersection": len(players_xi & all_bbb_players),
        "Missing on Right": len(players_xi - all_bbb_players),
        "Orphan Right IDs": len(all_bbb_players - players_xi)
    },
    {
        "Join Relationship": "mapping.player_id <-> playing_eleven.player_id",
        "Left Count": len(players_map),
        "Right Count": len(players_xi),
        "Intersection": len(players_map & players_xi),
        "Missing on Right": len(players_map - players_xi),
        "Orphan Right IDs": len(players_xi - players_map)
    },
    {
        "Join Relationship": "teams.team_id <-> matches home/away teams",
        "Left Count": len(team_ids_teams),
        "Right Count": len(team_ids_matches),
        "Intersection": len(team_ids_teams & team_ids_matches),
        "Missing on Right": len(team_ids_teams - team_ids_matches),
        "Orphan Right IDs": len(team_ids_matches - team_ids_teams)
    }
]

df_joins = pd.DataFrame(join_checks)
display(df_joins)

,Join Relationship,Left Count,Right Count,Intersection,Missing on Right,Orphan Right IDs
0,matches.match_id <-> ball_by_ball.match_id,883,867,867,16,0
1,matches.match_id <-> playing_eleven.match_id,883,880,880,3,0
2,playing_eleven.player_id <-> ball_by_ball players,1082,958,956,126,2
3,mapping.player_id <-> playing_eleven.player_id,1217,1082,1082,135,0
4,teams.team_id <-> matches home/away teams,38,38,38,0,0


---
## 6. Quick Reference Takeaways
Key observations for subsequent analysis and model feature engineering:

1. **File Hierarchy & Foreign Keys**:
   - `competitions` (`comp_id`) $\to$ `matches` (`match_id`, `comp_id`) $\to$ `ball_by_ball` (`match_id`)
   - `matches` $\to$ `playing_eleven` (`match_id`, `team_id`, `player_id`)
   - `teams` (`team_id`, `comp_id`) $\leftrightarrow$ `competition_team_player_mapping` (`comp_id`, `team_id`, `player_id`)
2. **Deliveries & Runs Semantics**:
   - `runs` represents batter runs scored off the bat.
   - `extras1_runs` and `extras_2_runs` contain extras conceded on the delivery.
   - Total team runs on a delivery is `runs + extras1_runs + extras_2_runs`.
3. **Extras & Legal Deliveries**:
   - `extras_type` includes `wide`, `no ball`, `leg bye`, `bye`, etc.
   - Deliveries labeled `wide` or `no ball` are illegal deliveries that do not consume an official ball in the over.
4. **Missing Values**:
   - `revised_target` and `revised_overs` in `matches.csv` are populated primarily for rain-affected / DLS matches.
   - Player dismissal fields in `ball_by_ball.csv` (`dismissed_player_id`, `dismissal_type`) are only populated on wicket deliveries.
5. **Referential Integrity**:
   - Ball-by-ball deliveries cleanly link back to valid matches and playing XI rosters without orphan matches.
